# 🚀 Notebook 05 — Veien videre

> *Amira:* «Nå har dere bygget, evaluert og besluttet. Dere forstår signaler, baselines,
> collaborative filtering, hybrider og fairness. Det er et solid fundament.
> Men feltet beveger seg fort. Her er seks retninger dere kan utforske på egenhånd.»

Denne notebooken inneholder **ingen kode** — bare pekere, intuisjoner og referanser.
Velg **én** retning som passer til ditt neste prosjekt, og gå dypere der.

---

## 1. Learning-to-Rank (LTR)

**Hva er dette?** I workshopen trente vi ALS med en implisitt feedback-loss.
Men ALS optimerer for *rekonstruksjon* av matrisen — ikke direkte for *rekkefølgen* av items i en liste.
Learning-to-Rank-metoder optimerer eksplisitt for rangeringskvalitet.

Det finnes tre familier:

- **Pointwise:** Behandler hvert item uavhengig — prediker en score, sorter etterpå. Enklest, men ignorerer at rangering er relativ.
- **Pairwise:** Optimerer for at *riktig* item skal scores høyere enn *feil* item. BPR (Bayesian Personalized Ranking) er den klassiske metoden her og brukes ofte som drop-in-erstatning for ALS-loss.
- **Listwise:** Optimerer hele listen som en enhet — f.eks. LambdaMART, som direkte optimerer NDCG. Kraftig, men krever mer data og mer tuning.

**Relevant for StreamNord:** Vi målte Recall og NDCG, men trente ikke *for* NDCG.
LTR lukker dette gapet — modellen lærer å rangere slik vi evaluerer.

### Referanser

- Rendle et al. (2009): *BPR: Bayesian Personalized Ranking from Implicit Feedback*
- Burges (2010): *From RankNet to LambdaRank to LambdaMART: An Overview*
- `LightFM`-biblioteket — implementerer BPR-loss med metadata-features

---

## 2. Bandit-algoritmer og utforskning

**Hva er dette?** Alle modellene vi testet har et blindpunkt: de anbefaler bare items
de allerede *tror* brukeren liker. Men hva om brukeren ville elsket noe modellen aldri
har sett dem interagere med? Dette er explore-exploit-dilemmaet.

Bandit-algoritmer balanserer mellom:

- **Exploit:** Anbefal det du vet fungerer (høy recall, trygt)
- **Explore:** Prøv noe nytt for å lære mer om brukerens smak (potensielt høyere gevinst)

Klassiske tilnærminger:

- **Epsilon-greedy:** Med sannsynlighet ε, anbefal et tilfeldig item. Enkelt, men blindt.
- **UCB (Upper Confidence Bound):** Anbefal items med høy *usikkerhet* — der modellen trenger mer data.
- **Thompson Sampling:** Sample fra modellens usikkerhet direkte. Elegant og effektivt.
- **Contextual Bandits:** Bruk brukerens kontekst (tid, enhet, sesjon) til å velge mellom explore og exploit.

**Relevant for StreamNord:** Cold-start-kurven i notebook 04 viste at ALS trenger historikk.
Bandits er en systematisk måte å samle den historikken raskere — spesielt for nye brukere og nye items.

### Referanser

- Li et al. (2010): *A Contextual-Bandit Approach to Personalized News Article Recommendation*
- Chapelle & Li (2011): *An Empirical Evaluation of Thompson Sampling*
- `vowpalwabbit` — industristandard for contextual bandits

---

## 3. Deep learning for anbefalinger

**Hva er dette?** ALS er en lineær modell — den finner brukere og items i et felles vektorrom,
men kan bare fange lineære mønstre. Nevrale nettverk kan lære mer komplekse interaksjoner.

To hovedretninger:

- **Neural Collaborative Filtering (NCF):** Erstatter det lineære prikk-produktet i ALS med et nevralt nettverk. Kan fange ikke-lineære bruker-item-interaksjoner, men trenger mer data og tuning enn ALS.
- **Sekvensielle modeller (SASRec, BERT4Rec):** Bruker transformer-arkitekturen til å modellere *rekkefølgen* av interaksjoner. I stedet for å behandle alle klikk som en uordnet mengde (slik ALS gjør), lærer disse modellene temporale mønstre — «brukere som ser A og deretter B, vil ofte se C».

**Når slår deep learning ALS?** Typisk når du har:

- Mye data (>10M interaksjoner)
- Rike features (tekst, bilder, metadata)
- Sekvensielle mønstre som er viktige (nyheter, musikk, video)

For datasett på størrelse med MovieLens 25M er forskjellen ofte overraskende liten.
ALS er enklere å drifte, raskere å trene, og lettere å debugge.

**Relevant for StreamNord:** Sesjonskonteksten i notebook 03 var en enkel Markov-kjede.
Transformer-baserte modeller (SASRec) ville fange langt rikere sekvensielle mønstre.

### Referanser

- He et al. (2017): *Neural Collaborative Filtering*
- Kang & McAuley (2018): *Self-Attentive Sequential Recommendation (SASRec)*
- Sun et al. (2019): *BERT4Rec: Sequential Recommendation with Bidirectional Encoder Representations*
- `RecBole`-biblioteket — unified framework for 70+ recommendation-modeller

---

## 4. Kunnskapsgrafer og grafnevrale nettverk (GNNs)

**Hva er dette?** ALS ser bare én type relasjon: bruker → item.
Men i virkeligheten finnes det rike relasjoner: regissør, skuespiller, studio, sjanger,
«lignende som», «oppfølger av». En kunnskapsgraf modellerer alt dette som et nettverk.

Grafnevrale nettverk (GNNs) lærer representasjoner av noder (brukere, items, entiteter)
ved å aggregere informasjon fra naboer i grafen. To kjente tilnærminger:

- **PinSage (Pinterest):** Bruker random walks og GNN-lag for å lære item-embeddings fra en graf med milliarder av noder. Designet for industriell skala.
- **KGAT (Knowledge Graph Attention Network):** Bruker attention-mekanismer for å vekte ulike typer relasjoner ulikt — en regissør-relasjon kan være viktigere enn en studio-relasjon.

**Relevant for StreamNord:** Metadata i notebook 01 var flat (genre-vektorer).
En kunnskapsgraf ville beriket dette med regissører, skuespillere og tematiske koblinger
— og potensielt gitt bedre forklaringer: «Anbefalt fordi du liker filmer av Christopher Nolan».

### Referanser

- Ying et al. (2018): *Graph Convolutional Neural Networks for Web-Scale Recommender Systems (PinSage)*
- Wang et al. (2019): *KGAT: Knowledge Graph Attention Network for Recommendation*
- `PyTorch Geometric` og `DGL` — biblioteker for grafnevrale nettverk

---

## 5. Reinforcement learning for anbefalinger

**Hva er dette?** Alle modellene våre optimerer for *neste klikk*.
Men hva om det beste valget nå er å vise noe som gir *langsiktig* verdi?
Reinforcement learning (RL) behandler anbefaling som en sekvensiell beslutningsprosess
der målet er å maksimere total brukerverdi over tid.

I praksis betyr dette:

- **State:** Brukerens historikk og kontekst
- **Action:** Hvilke items å anbefale
- **Reward:** Klikk, avspillingstid, abonnementsfornyelse
- **Policy:** En strategi som velger handlinger for å maksimere langsiktig belønning

Utfordringene er betydelige:

- **Off-policy evaluering:** Du kan ikke teste en ny policy uten å rulle den ut — men du vil gjerne evaluere den først.
- **Reward shaping:** Hva er egentlig «god» brukeropplevelse over tid? Klikk ≠ tilfredshet.
- **Skalering:** RL-trening er ustabil og ressurskrevende sammenlignet med ALS.

**Relevant for StreamNord:** Bandits (seksjon 2) er et enklere spesialtilfelle av RL.
Full RL blir relevant når StreamNord vil optimere for *retention* (beholde abonnenter),
ikke bare *engagement* (klikk). Det er et viktig skille.

### Referanser

- Afsar et al. (2022): *Reinforcement Learning based Recommender Systems: A Survey*
- Chen et al. (2019): *Top-K Off-Policy Correction for a REINFORCE Recommender System (YouTube)*
- Ie et al. (2019): *SlateQ: A Tractable Decomposition for Reinforcement Learning with Recommendation Sets*

---

## 6. LLM-baserte anbefalinger

**Hva er dette?** Store språkmodeller (LLMs) endrer hvordan vi tenker om anbefalinger.
I stedet for å lære embeddings fra interaksjonsdata alene, kan en LLM bruke
*tekstbeskrivelser* av items og brukere til å forstå preferanser.

Tre fremvoksende bruksområder:

- **Item-forståelse:** Bruk en LLM til å lese filmbeskrivelser, anmeldelser og metadata og generere rike embeddings. Erstatter manuelle genre-vektorer med semantisk forståelse.
- **Forklaringsgenerering:** I stedet for «Anbefalt fordi du likte X» (template-basert), kan en LLM generere naturlige, kontekstuelle forklaringer: «Denne thrilleren har samme tematikk som filmene du så forrige uke».
- **Samtalebasert anbefaling:** Brukeren *snakker* med systemet: «Jeg vil se noe som Inception, men mer saktegående». LLM-en tolker intensjon og filtrerer kandidater.

**Utfordringer:**

- Latency: LLM-inference er treg sammenlignet med et ALS-prikk-produkt
- Hallusinasjon: LLM-en kan anbefale filmer som ikke finnes
- Evaluering: Tradisjonelle metrikker (Recall, NDCG) fanger ikke kvaliteten på forklaringer

**Relevant for StreamNord:** Skriveøvelsen i notebook 04 — «forklar uten fagspråk» — er
akkurat det en LLM kan automatisere. Og samtalebasert anbefaling er kanskje den mest
naturlige brukeropplevelsen for oppdagelse av nytt innhold.

### Referanser

- Wu et al. (2024): *A Survey on Large Language Models for Recommendation*
- Gao et al. (2023): *Chat-REC: Towards Interactive and Explainable LLMs-Augmented Recommender System*
- Bao et al. (2023): *TALLRec: An Effective and Efficient Tuning Framework to Align Large Language Models with Recommendation*

---

## Hva bør du velge?

| Hvis du jobber med … | Start her |
|---|---|
| Implisitt feedback og vil forbedre ALS | **Learning-to-Rank** (BPR) |
| Nye brukere/items og trenger rask læring | **Bandits** |
| Sekvensielle data (nyheter, musikk, video) | **Deep learning** (SASRec) |
| Rike metadata og relasjoner | **Kunnskapsgrafer** |
| Langsiktig brukerverdi, ikke bare klikk | **Reinforcement learning** |
| Naturlig språk, forklaringer, samtale | **LLM-baserte anbefalinger** |

> *Amira:* «Velg én ting. Bygg en prototype. Mål den. Så vet du om det er verdt å investere.»